# Gemma 4 E2B Fine-Tuning — Mininio Carb Assistant

Thin Colab wrapper. All logic lives in `finetuning/gemma/train_gemma.py`.

**Prerequisites:**
1. Upload `data/output/gemma/` (train.jsonl + eval.jsonl) in the same `mininio-data.zip` to your Drive root
2. The zip contents should be `gemma/train.jsonl` and `gemma/eval.jsonl` (no extra wrapper directory)
3. Create with: `cd data/output && zip -r mininio-data.zip lfm/ gemma/`
4. Accept Gemma license at https://huggingface.co/unsloth/gemma-4-E2B-it
5. Set HuggingFace token in Colab secrets (HF_TOKEN) or provide below

**Runtime:** Colab Free T4 (16GB) — ~60-120 min for 3 epochs
**Note:** Initial loss ~13-15 is NORMAL for Gemma 4 E2B — do not panic.
---

In [ ]:
# Install dependencies (Colab-specific pins from Unsloth reference)
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -q unsloth
else:
    import torch
    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v == "2.9" else "0.0.32.post2" if v == "2.8" else "0.0.29.post3")
    !pip install -q --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install -q --no-deps unsloth

!pip install -q loguru

os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")
print("Dependencies installed.")

In [ ]:
# Clone repo & mount Drive, copy data
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/mininio-ai-finetuning'
REPO_URL = 'https://github.com/CoGian/mininio-ai-finetuning.git'

if os.path.isdir(f'{REPO_DIR}/.git'):
    print('Pulling latest...')
    !git -C "{REPO_DIR}" pull
else:
    print('Cloning repo...')
    !git clone "{REPO_URL}" "{REPO_DIR}"

!cp /content/drive/MyDrive/mininio-data.zip /content/ 2>/dev/null
!mkdir -p /content/data/output
!unzip -o /content/mininio-data.zip -d /content/data/output/ 2>/dev/null
!ls /content/data/output/gemma/ 2>/dev/null || echo "Data not found. Make sure mininio-data.zip is in Drive root."

In [ ]:
# Run training (3 epochs, max_seq_length 4096, QLoRA r16, effective batch 8)
!PYTHONPATH=/content/mininio-ai-finetuning python -m finetuning.gemma.train_gemma \
    --data-dir /content/data/output \
    --output-dir /content/drive/MyDrive/mininio-checkpoints \
    --epochs 3 \
    --max-seq-length 4096 \
    --batch-size 2 \
    --grad-accum 4 \
    --report-to none

print("\nCheckpoints saved to Drive: /content/drive/MyDrive/mininio-checkpoints/gemma/")

### Smoke Test (optional)

In [ ]:
# Quick smoke test
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name="/content/drive/MyDrive/mininio-checkpoints/gemma/merged_16bit",
    max_seq_length=4096,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
FastModel.for_inference(model)

messages = [{"role": "user", "content": "I ate 100g of potatoes. How many carbs?"}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    use_cache=True,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)